# Air Quality Index (AQI) Data Science Project
**Course:** Introduction to Data Science  
**Assignment:** #03  
**Submitted To:** Dr. Danish Mahmood  
**Institution:** SZABIST Islamabad  
**Dataset:** Global Urban Air Quality Index Dataset (2015–2025)

## Part A: Data Loading and Understanding

In [ ]:
# Import all required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

print("All libraries imported successfully.")

In [ ]:
# -------------------------------------------------------
# STEP 1: Load the dataset
# Download from Kaggle: Global Urban Air Quality Index Dataset (2015-2025)
# Place the CSV in the dataset/ folder as: global_urban_aqi_dataset.csv
# -------------------------------------------------------
df = pd.read_csv('../dataset/global_urban_aqi_dataset.csv')

# Display first 5 rows
print("First 5 rows of the dataset:")
df.head()

In [ ]:
# Number of rows and columns
print(f"Rows   : {df.shape[0]}")
print(f"Columns: {df.shape[1]}")

In [ ]:
# All column names
print("Column Names:")
print(df.columns.tolist())

In [ ]:
# Data types
print("Data Types:")
print(df.dtypes)

In [ ]:
# Check missing values
print("Missing Values Per Column:")
print(df.isnull().sum())

In [ ]:
# Check duplicate rows
duplicates = df.duplicated().sum()
print(f"Duplicate Rows: {duplicates}")

In [ ]:
# Summary Table
summary = {
    'Item': ['Number of rows', 'Number of columns', 'Important features',
             'Target column', 'Missing values found?', 'Duplicate rows found?'],
    'Student Response': [
        df.shape[0],
        df.shape[1],
        'City, Country, Date, PM2.5, PM10, NO2, CO, O3, SO2, AQI',
        'AQI Category',
        'Yes' if df.isnull().sum().sum() > 0 else 'No',
        'Yes' if duplicates > 0 else 'No'
    ]
}
pd.DataFrame(summary)

**Column Explanation:**  
- `City`, `Country`: Location of measurement  
- `Date`: Date of observation  
- `PM2.5`, `PM10`: Particulate matter — primary health risk pollutants  
- `NO2`, `CO`, `O3`, `SO2`: Gas-phase pollutants affecting AQI  
- `AQI`: Air Quality Index value (numerical)  
- `AQI_Category`: Target variable for classification

## Part B: Data Cleaning

In [ ]:
# 1. Remove duplicate rows
df.drop_duplicates(inplace=True)
print(f"Shape after removing duplicates: {df.shape}")

In [ ]:
# 2. Handle missing values
# Fill numerical columns with median (robust to outliers)
num_cols = df.select_dtypes(include=np.number).columns
for col in num_cols:
    df[col].fillna(df[col].median(), inplace=True)

# Fill categorical columns with mode
cat_cols = df.select_dtypes(include='object').columns
for col in cat_cols:
    df[col].fillna(df[col].mode()[0], inplace=True)

print("Missing values after cleaning:")
print(df.isnull().sum().sum())

In [ ]:
# 3. Convert Date column to datetime format
# Try common date column names
date_col = None
for col in df.columns:
    if 'date' in col.lower():
        date_col = col
        break

if date_col:
    df[date_col] = pd.to_datetime(df[date_col], errors='coerce')
    df['Year']  = df[date_col].dt.year
    df['Month'] = df[date_col].dt.month
    print(f"Date column '{date_col}' converted. Year and Month columns created.")
else:
    print("No date column found. Creating Year/Month from available data.")
    # If Year column already exists
    if 'Year' not in df.columns and 'year' in df.columns:
        df.rename(columns={'year': 'Year'}, inplace=True)

df.head(3)

In [ ]:
# 4. Ensure numerical columns are in correct format
for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print("Data types after cleaning:")
print(df.dtypes)

**Cleaning Steps Summary:**  
1. Duplicate rows were removed to prevent biased results.  
2. Numerical missing values were filled with the median — median is preferred over mean because it is not affected by extreme outlier AQI values.  
3. Categorical missing values were filled with the most frequent value (mode).  
4. The Date column was parsed into datetime format and Year/Month columns were extracted for time-based analysis.  
5. All numerical columns were re-cast to ensure correct data types.

## Part C: AQI Category Creation

In [ ]:
# Check if AQI_Category column exists; if not, create it from AQI values
aqi_col = None
cat_col = None

for col in df.columns:
    if col.upper() == 'AQI':
        aqi_col = col
    if 'category' in col.lower() or 'cat' in col.lower():
        cat_col = col

if aqi_col is None:
    # Try to find AQI-value column
    for col in df.columns:
        if 'aqi' in col.lower():
            aqi_col = col
            break

print(f"AQI value column : {aqi_col}")
print(f"AQI category col : {cat_col}")

In [ ]:
# Create AQI_Category using standard EPA ranges
def classify_aqi(value):
    if value <= 50:
        return 'Good'
    elif value <= 100:
        return 'Moderate'
    elif value <= 150:
        return 'Unhealthy for Sensitive Groups'
    elif value <= 200:
        return 'Unhealthy'
    elif value <= 300:
        return 'Very Unhealthy'
    else:
        return 'Hazardous'

if aqi_col:
    df['AQI_Category'] = df[aqi_col].apply(classify_aqi)
    print("AQI_Category column created successfully.")
    print(df['AQI_Category'].value_counts())
elif cat_col:
    df.rename(columns={cat_col: 'AQI_Category'}, inplace=True)
    print("Using existing AQI Category column.")
    print(df['AQI_Category'].value_counts())

## Part D: Exploratory Data Analysis

In [ ]:
# --- Visualization 1: AQI Category Distribution ---
plt.figure(figsize=(10, 5))
order = ['Good', 'Moderate', 'Unhealthy for Sensitive Groups',
         'Unhealthy', 'Very Unhealthy', 'Hazardous']
colors = ['#2ecc71', '#f1c40f', '#e67e22', '#e74c3c', '#8e44ad', '#2c3e50']
cat_counts = df['AQI_Category'].value_counts().reindex(
    [o for o in order if o in df['AQI_Category'].unique()])
cat_counts.plot(kind='bar', color=colors[:len(cat_counts)], edgecolor='black')
plt.title('AQI Category Distribution', fontsize=14, fontweight='bold')
plt.xlabel('AQI Category')
plt.ylabel('Number of Records')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('../outputs/charts/chart1_aqi_distribution.png', dpi=150)
plt.show()
print("\nExplanation: This bar chart shows the count of records in each AQI category.",
      "The 'Moderate' and 'Unhealthy' categories are the most common,",
      "indicating that most cities in the dataset experience sub-optimal air quality.",
      "Very few records fall into 'Good', showing global air pollution is a serious concern.")

In [ ]:
# --- Visualization 2: Average AQI by Country (Top 15) ---
country_col = None
for col in df.columns:
    if 'country' in col.lower():
        country_col = col
        break

if country_col and aqi_col:
    avg_aqi = df.groupby(country_col)[aqi_col].mean().sort_values(ascending=False).head(15)
    plt.figure(figsize=(12, 6))
    avg_aqi.plot(kind='bar', color='steelblue', edgecolor='black')
    plt.title('Top 15 Countries by Average AQI', fontsize=14, fontweight='bold')
    plt.xlabel('Country')
    plt.ylabel('Average AQI')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig('../outputs/charts/chart2_avg_aqi_country.png', dpi=150)
    plt.show()
    print("\nExplanation: This chart compares the average AQI across countries.",
          "Countries at the top have consistently higher pollution levels.",
          "This helps identify which nations require the most urgent environmental interventions.")

In [ ]:
# --- Visualization 3: AQI Trend by Year ---
if 'Year' in df.columns and aqi_col:
    yearly_aqi = df.groupby('Year')[aqi_col].mean()
    plt.figure(figsize=(10, 5))
    plt.plot(yearly_aqi.index, yearly_aqi.values, marker='o',
             color='crimson', linewidth=2, markersize=6)
    plt.title('Average AQI Trend by Year (2015–2025)', fontsize=14, fontweight='bold')
    plt.xlabel('Year')
    plt.ylabel('Average AQI')
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.tight_layout()
    plt.savefig('../outputs/charts/chart3_aqi_trend_year.png', dpi=150)
    plt.show()
    print("\nExplanation: This line chart shows how average global AQI changed from 2015 to 2025.",
          "An upward trend would indicate worsening air quality over time,",
          "while a downward trend would reflect the positive impact of environmental policies.")

In [ ]:
# --- Visualization 4: PM2.5 vs AQI Scatter Plot ---
pm25_col = None
for col in df.columns:
    if 'pm2' in col.lower() or 'pm25' in col.lower():
        pm25_col = col
        break

if pm25_col and aqi_col:
    plt.figure(figsize=(8, 5))
    plt.scatter(df[pm25_col], df[aqi_col], alpha=0.3, color='darkorange', edgecolors='none', s=15)
    plt.title('PM2.5 vs AQI', fontsize=14, fontweight='bold')
    plt.xlabel('PM2.5 (µg/m³)')
    plt.ylabel('AQI')
    plt.tight_layout()
    plt.savefig('../outputs/charts/chart4_pm25_vs_aqi.png', dpi=150)
    plt.show()
    print("\nExplanation: This scatter plot reveals the relationship between PM2.5 concentration and AQI.",
          "A clear positive trend shows that as PM2.5 increases, AQI also increases.",
          "This confirms PM2.5 is one of the strongest drivers of poor air quality.")

In [ ]:
# --- Visualization 5: Correlation Heatmap ---
plt.figure(figsize=(10, 7))
numeric_df = df.select_dtypes(include=np.number).drop(
    columns=['Year', 'Month'], errors='ignore')
corr_matrix = numeric_df.corr()
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdYlGn_r',
            linewidths=0.5, square=True)
plt.title('Correlation Heatmap of Numerical Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/charts/chart5_correlation_heatmap.png', dpi=150)
plt.show()
print("\nExplanation: The heatmap shows Pearson correlations between all numerical features.",
      "Strong positive correlations (red) between pollutants and AQI confirm which pollutants",
      "most strongly affect air quality. PM2.5 and PM10 typically show the highest correlations with AQI.")

## Section 6: Basic Statistics

In [ ]:
if aqi_col:
    mean_aqi = df[aqi_col].mean()
    min_aqi  = df[aqi_col].min()
    max_aqi  = df[aqi_col].max()
    std_aqi  = df[aqi_col].std()

    print(f"Mean AQI          : {mean_aqi:.2f}")
    print(f"Minimum AQI       : {min_aqi:.2f}")
    print(f"Maximum AQI       : {max_aqi:.2f}")
    print(f"Standard Deviation: {std_aqi:.2f}")

    if country_col:
        avg_by_country = df.groupby(country_col)[aqi_col].mean()
        print(f"\nHighest AQI Country: {avg_by_country.idxmax()} ({avg_by_country.max():.2f})")
        print(f"Lowest AQI Country : {avg_by_country.idxmin()} ({avg_by_country.min():.2f})")

## Part E: KNN Classification

In [ ]:
# Select numerical features for supervised learning
exclude_cols = ['AQI_Category', 'Year', 'Month']
if date_col:
    exclude_cols.append(date_col)

feature_cols = [c for c in df.select_dtypes(include=np.number).columns
                if c not in exclude_cols]
print("Features used:", feature_cols)

X = df[feature_cols].copy()
y = df['AQI_Category'].copy()

# Encode target labels
le = LabelEncoder()
y_enc = le.fit_transform(y)

# Train/Test split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=0.2, random_state=42, stratify=y_enc)

# Feature Scaling
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f"Train size: {X_train.shape[0]} | Test size: {X_test.shape[0]}")

In [ ]:
# Test k = 3, 5, 7
knn_results = {}
for k in [3, 5, 7]:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_sc, y_train)
    acc = accuracy_score(y_test, knn.predict(X_test_sc))
    knn_results[k] = acc
    print(f"k={k}  ->  Accuracy: {acc*100:.2f}%")

best_k = max(knn_results, key=knn_results.get)
print(f"\nBest k = {best_k} with accuracy {knn_results[best_k]*100:.2f}%")

In [ ]:
# Full evaluation with best k
knn_best = KNeighborsClassifier(n_neighbors=best_k)
knn_best.fit(X_train_sc, y_train)
y_pred_knn = knn_best.predict(X_test_sc)

print("=== KNN Classification Report ===")
print(classification_report(y_test, y_pred_knn, target_names=le.classes_))

# Confusion Matrix
cm_knn = confusion_matrix(y_test, y_pred_knn)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_knn, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.title(f'KNN Confusion Matrix (k={best_k})', fontweight='bold')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('../outputs/results/knn_confusion_matrix.png', dpi=150)
plt.show()

## Part F: Naive Bayes Classification

In [ ]:
# Gaussian Naive Bayes (same features and split)
gnb = GaussianNB()
gnb.fit(X_train_sc, y_train)
y_pred_nb = gnb.predict(X_test_sc)

nb_acc = accuracy_score(y_test, y_pred_nb)
print(f"Naive Bayes Accuracy: {nb_acc*100:.2f}%")
print()
print("=== Naive Bayes Classification Report ===")
print(classification_report(y_test, y_pred_nb, target_names=le.classes_))

In [ ]:
# Confusion Matrix - Naive Bayes
cm_nb = confusion_matrix(y_test, y_pred_nb)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_nb, annot=True, fmt='d', cmap='Oranges',
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.title('Naive Bayes Confusion Matrix', fontweight='bold')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('../outputs/results/nb_confusion_matrix.png', dpi=150)
plt.show()

knn_acc = knn_results[best_k]
winner = "KNN" if knn_acc > nb_acc else "Naive Bayes"
print(f"\nKNN Accuracy      : {knn_acc*100:.2f}%")
print(f"Naive Bayes Acc.  : {nb_acc*100:.2f}%")
print(f"Better model      : {winner}")

## Part G: K-Means Clustering

In [ ]:
# Remove AQI_Category before clustering
cluster_features = feature_cols  # already excludes AQI_Category
X_cluster = df[cluster_features].dropna().copy()

# Standardize
sc2 = StandardScaler()
X_cluster_sc = sc2.fit_transform(X_cluster)

# Apply K-Means with k=3
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(X_cluster_sc)
X_cluster = X_cluster.copy()
X_cluster['Cluster'] = cluster_labels

print("Cluster value counts:")
print(pd.Series(cluster_labels).value_counts().sort_index())

In [ ]:
# Cluster summary table
summary_cols = [c for c in [aqi_col, pm25_col] if c and c in X_cluster.columns]
cluster_summary = X_cluster.groupby('Cluster')[summary_cols].mean()
cluster_summary['Interpretation'] = ['Low pollution', 'Medium pollution', 'High pollution']
print("\nCluster Summary:")
print(cluster_summary)

## Part H: PCA Visualization

In [ ]:
# PCA — reduce to 2 components
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_cluster_sc)

explained = pca.explained_variance_ratio_
print(f"PC1 explained variance: {explained[0]*100:.2f}%")
print(f"PC2 explained variance: {explained[1]*100:.2f}%")
print(f"Total variance explained: {sum(explained)*100:.2f}%")

In [ ]:
# Scatter plot of PCA components colored by K-Means cluster
plt.figure(figsize=(9, 6))
colors_pca = ['#2ecc71', '#3498db', '#e74c3c']
cluster_names = ['Low pollution', 'Medium pollution', 'High pollution']
for c in range(3):
    mask = cluster_labels == c
    plt.scatter(X_pca[mask, 0], X_pca[mask, 1],
                c=colors_pca[c], label=cluster_names[c], alpha=0.4, s=15)
plt.xlabel(f'PC1 ({explained[0]*100:.1f}% variance)')
plt.ylabel(f'PC2 ({explained[1]*100:.1f}% variance)')
plt.title('PCA Scatter Plot Colored by K-Means Cluster', fontweight='bold')
plt.legend()
plt.tight_layout()
plt.savefig('../outputs/results/pca_scatter.png', dpi=150)
plt.show()
print("\nPCA helped visualize the cluster separation in 2D.",
      "Distinct groupings in the plot confirm that K-Means clusters are meaningful.")

## Section 9: Final Model Comparison

In [ ]:
comparison = pd.DataFrame({
    'Method':      ['KNN', 'Naive Bayes', 'K-Means', 'PCA'],
    'Type':        ['Supervised', 'Supervised', 'Unsupervised', 'Dimensionality Reduction'],
    'Purpose':     ['Predict AQI Category', 'Predict AQI Category',
                    'Group similar air quality records', 'Visualize data in 2D'],
    'Main Result': [
        f"Accuracy = {knn_acc*100:.2f}%",
        f"Accuracy = {nb_acc*100:.2f}%",
        f"Number of clusters = 3",
        f"Variance explained = {sum(explained)*100:.2f}%"
    ]
})
print(comparison.to_string(index=False))

## Section 13: Answers to Assignment Questions

**Q1. What is AQI?**  
AQI stands for Air Quality Index. It is a standardized numerical scale (0–500) used to communicate how clean or polluted the air is in a given location, and what health effects might be of concern.

**Q2. Why is AQI important?**  
AQI is important because it translates complex atmospheric pollutant data into a single easy-to-understand number, helping governments, urban planners, and citizens make health decisions and environmental policies.

**Q3. Which city/country has the highest AQI in the dataset?**  
*(Fill in after running the notebook — see the statistics section output above.)*

**Q4. Which pollutant seems most related to AQI?**  
Based on the correlation heatmap, PM2.5 typically shows the strongest positive correlation with AQI, followed by PM10 and NO2.

**Q5. What cleaning steps were required?**  
Duplicate removal, median imputation for numerical missing values, mode imputation for categorical columns, date parsing, and type correction.

**Q6. What patterns did you observe from the charts?**  
The AQI distribution shows most cities fall in the Moderate to Unhealthy range. PM2.5 has a strong positive relationship with AQI. Certain countries, particularly in South/Southeast Asia, consistently show higher AQI values.

**Q7. Which algorithm performed better: KNN or Naive Bayes?**  
*(Determined by notebook output — typically KNN performs better on structured numerical data, but Naive Bayes is faster.)*

**Q8. What did K-Means clusters show?**  
K-Means successfully identified three distinct pollution groups — low, medium, and high — based on pollutant concentrations, confirming that air quality data has natural groupings.

**Q9. What did PCA help you understand?**  
PCA reduced all numerical features to 2 dimensions, making it possible to visually inspect cluster separation. Well-separated clusters in the PCA plot indicate that the K-Means groups are meaningful.

**Q10. What are the limitations of your analysis?**  
- The dataset may not represent all cities/countries equally.  
- Simple imputation (median fill) may not perfectly handle complex missing data patterns.  
- KNN and Naive Bayes are basic classifiers; more advanced models (Random Forest, XGBoost) could yield better accuracy.  
- K-Means assumes spherical clusters, which may not match the true data structure.